# RoBERTa

In [ ]:
# argument classification with RoBERTa

from transformers import pipeline, AutoTokenizer
from collections import Counter
import pandas as pd

classifier = pipeline("zero-shot-classification", model="roberta-large-mnli")
tokenizer = AutoTokenizer.from_pretrained("roberta-large-mnli")

annotations_df = pd.read_csv("Testing Dataset - Final.csv")
df = pd.read_csv("comments_definitive.csv")

comment_ids = annotations_df['ID'].dropna().astype(int).tolist()

candidate_labels = [
    "1.1 Not all PFAS are dangerous - the argument cites polymers of low concern (PLC).",
    "1.2 A given PFAS is claimed to be not toxic.",
    "2.1 There is no alternative to a given PFAS.",
    "2.2 Existing alternatives perform worse than PFAS-based materials.",
    "3.1 Negative economic impact on business from the PFAS restriction.",
    "3.2 Production is forced to move out of Europe.",
    "3.3 Production job loss because of the restrictions.",
    "3.4 Loss of EU competitiveness and strategic autonomy.",
    "4.1 Negative impact on customer convenience and the modern life if PFAS is banned.",
    "4.2 Negative impact on the health and safety of citizens if PFAS is banned.",
    "4.3 PFAS are needed to achieve Green transition because of their efficiency to minimise emissions (Green Deal argument).",
    "5. Unclear, no arguments or in favour of banning PFAS"
]
label_code_map = {lbl: lbl.split(" ")[0] for lbl in candidate_labels}

from tqdm import tqdm
import time

comment_ids = annotations_df['ID'].dropna().astype(int).tolist()
classification_results = {}

print(f"Running RoBERTa classification on {len(comment_ids)} comments...")
start_time = time.time()

for target_id in tqdm(comment_ids, desc="Classifying"):
    df_selected = df[(df['ID'] == target_id) & df['Combined Comments'].notna()]
    if df_selected.empty:
        classification_results[target_id] = "5"
        continue

    full_text = df_selected.iloc[0]['Combined Comments']
    tokens = tokenizer.encode(full_text, add_special_tokens=False)
    chunks = [tokens[i:i + 500] for i in range(0, len(tokens), 500)]
    chunk_texts = [tokenizer.decode(chunk, skip_special_tokens=True) for chunk in chunks]

    label_votes = Counter()
    for chunk_text in chunk_texts:
        try:
            result = classifier(chunk_text, candidate_labels, multi_label=True)
            for label, score in zip(result['labels'], result['scores']):
                if score >= 0.5:
                    label_votes[label_code_map[label]] += 1
        except Exception:
            classification_results[target_id] = "ERROR"
            break

    if target_id not in classification_results:
        codes = sorted(label_votes.keys())
        classification_results[target_id] = ",".join(codes) if codes else "5"

end_time = time.time()
print(f"\n✅ Classification complete in {end_time - start_time:.2f} seconds.")


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report

response_df = pd.DataFrame.from_dict(classification_results, orient='index', columns=["Response"]).reset_index().rename(columns={"index": "ID"})
df_merged = annotations_df.merge(response_df, on="ID", how="left")
df_merged["Response"] = df_merged["Response"].fillna("5").str.replace(" ", "")

# dummifying prediction labels
pred_dummies = df_merged["Response"].str.get_dummies(sep=",")
pred_dummies.columns = [f"{col}_pred" for col in pred_dummies.columns]
df_merged = pd.concat([df_merged, pred_dummies], axis=1)

pred_cols = [col for col in df_merged.columns if col.endswith("_pred")]
non5_cols = [col for col in pred_cols if col != "5_pred"]
df_merged.loc[df_merged[non5_cols].any(axis=1), '5_pred'] = 0

# dummifying true labels
df_merged["Argument types"] = df_merged["Argument types"].fillna("5").astype(str).str.replace(" ", "")
eval_dummies = df_merged["Argument types"].str.get_dummies(sep=",")
eval_dummies.columns = [f"{col}_eval" for col in eval_dummies.columns]
df_merged = pd.concat([df_merged, eval_dummies], axis=1)

# evaluation
y_true = df_merged.filter(regex=r'^(\d\.\d|5)_eval$').sort_index(axis=1)
y_pred = df_merged.filter(regex=r'^(\d\.\d|5)_pred$').sort_index(axis=1)

print("=== Micro ===")
print("Precision:", precision_score(y_true, y_pred, average='micro'))
print("Recall:", recall_score(y_true, y_pred, average='micro'))
print("F1:", f1_score(y_true, y_pred, average='micro'))
print("Accuracy:", accuracy_score(y_true, y_pred))

print("\n=== Macro ===")
print("Precision:", precision_score(y_true, y_pred, average='macro'))
print("Recall:", recall_score(y_true, y_pred, average='macro'))
print("F1:", f1_score(y_true, y_pred, average='macro'))

